In [ ]:
# -*- coding: utf-8 -*-
"""
大规模中文语料清洗与质量评估（仅数据集）
自动检测文件编码，过滤非自然语言文本，输出高质量语料
"""

import os
import glob
import re
import hashlib
import random
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False


# ==================== 清洗器（总分0-3）====================
class CorpusCleaner:
    def __init__(self, exact_dedup=True, desensitize=True, sensitive_words=None, verbose=False):
        self.exact_dedup = exact_dedup
        self.desensitize = desensitize
        self.sensitive_words = set(sensitive_words or ["暴力", "色情"])
        self.verbose = verbose

    def exact_deduplication(self, df):
        before = len(df)
        df['hash'] = df['text'].apply(lambda x: hashlib.md5(str(x).encode('utf-8')).hexdigest())
        df = df.drop_duplicates(subset=['hash']).reset_index(drop=True)
        after = len(df)
        if self.verbose:
            print(f"精确去重：{before} -> {after} (去重率 {(1-after/before)*100:.2f}%)")
        df.drop(columns=['hash'], inplace=True)
        return df

    def desensitize_text(self, df):
        def desensitize(text):
            if not isinstance(text, str):
                return ""
            text = re.sub(r'\b1[3-9]\d{9}\b', '手机号', text)
            text = re.sub(r'\b[\w\.-]+@[\w\.-]+\.\w+\b', '邮箱', text)
            return text
        df['text_clean'] = df['text'].apply(desensitize)
        if self.verbose:
            print("脱敏完成")
        return df

    def quality_scoring(self, df):
        """评分范围 0-3，各维度权重：完整性(0-0.75)，流畅性(0-0.75)，安全性(0-0.75)，敏感词(0-0.75)"""
        def score(text):
            if not isinstance(text, str) or len(text) == 0:
                return (0, 0, 0, 0, 0)
            length = len(text)
            # 完整性 (0-0.75)
            if length < 10:
                comp = 0.0
            elif length < 30:
                comp = 0.375
            else:
                comp = 0.75
            # 流畅性 (0-0.75)
            flu = 0.75
            if re.search(r'(.)\1{10,}', text):
                flu -= 0.375
            garbage = re.findall(r'[^a-zA-Z\u4e00-\u9fff\s，。！？；：""''《》【】（）]', text)
            if len(garbage) > length * 0.3:
                flu -= 0.375
            if not re.search(r'[\u4e00-\u9fff]', text) and length > 100:
                flu -= 0.375
            flu = max(0.0, flu)
            # 安全性 (0-0.75)
            harmful = ["暴力", "色情", "诈骗"]
            safety = 0.0 if any(w in text for w in harmful) else 0.75
            # 敏感词 (0-0.75)
            sensitivity = 0.0 if any(w in text for w in self.sensitive_words) else 0.75
            total = comp + flu + safety + sensitivity
            return total, comp, flu, safety, sensitivity

        scores = df['text'].apply(score).apply(pd.Series)
        scores.columns = ['total_score', 'completeness', 'fluency', 'safety', 'sensitivity']
        df = pd.concat([df, scores], axis=1)
        if self.verbose:
            print(f"评分完成，平均分={df['total_score'].mean():.2f} (满分3)")
        return df

    def generate_report(self, df, high_threshold=2.0, output_dir='./output'):
        os.makedirs(output_dir, exist_ok=True)
        high = df[df['total_score'] >= high_threshold]
        low = df[df['total_score'] < high_threshold]
        if self.verbose:
            print(f"\n总样本: {len(df)}, 高质量(≥{high_threshold}): {len(high)}, 低质量: {len(low)}")
            print(f"高质量比例: {len(high)/len(df)*100:.2f}%")
            print("分数分布:")
            print(df['total_score'].value_counts().sort_index())
        if len(high) > 0:
            high[['text_clean', 'total_score']].to_json(
                os.path.join(output_dir, 'high_quality.json'),
                orient='records', force_ascii=False, indent=2
            )
        if len(low) > 0:
            low[['text', 'total_score']].to_csv(
                os.path.join(output_dir, 'low_quality.csv'),
                index=False, encoding='utf-8-sig'
            )
        if self.verbose:
            plt.figure(figsize=(8,4))
            plt.hist(df['total_score'], bins=20, color='skyblue', edgecolor='black')
            plt.xlabel('质量总分 (0-3)')
            plt.ylabel('样本数')
            plt.title('清洗后数据质量分布')
            plt.show()
        return high, low


# ==================== 数据加载器（仅真实数据）====================
class RealDataLoader:
    def __init__(self, verbose=False):
        self.verbose = verbose

    def _is_natural_language(self, text):
        """判断一行文本是否为自然语言（而非纯数字、标点、代码等）"""
        if not isinstance(text, str) or len(text) < 5 or len(text) > 5000:
            return False
        chinese_chars = re.findall(r'[\u4e00-\u9fff]', text)
        chinese_ratio = len(chinese_chars) / max(1, len(text))
        if chinese_ratio < 0.1 and len(re.findall(r'[^\w\s]', text)) > len(text) * 0.5:
            return False
        if re.match(r'^[\d\s\.,;:!?\-_=+*&^%$#@~`|\\/\[\]{}()<>]+$', text):
            return False
        return True

    def _read_file_with_encodings(self, file_path):
        """尝试多种编码读取文件，返回行列表"""
        encodings = ['utf-8', 'gbk', 'gb18030', 'latin-1']
        for enc in encodings:
            try:
                with open(file_path, 'r', encoding=enc) as f:
                    lines = f.readlines()
                return lines
            except (UnicodeDecodeError, UnicodeError):
                continue
        if self.verbose:
            print(f"无法读取文件（尝试了 {encodings}）: {file_path}")
        return []

    def _process_file(self, file_path):
        """读取单个文件，返回经过自然语言过滤的文本行列表"""
        lines = self._read_file_with_encodings(file_path)
        if not lines:
            return []
        valid_lines = []
        for line in lines:
            line = line.strip()
            if not line:
                continue
            # 处理 __label__ 标签格式
            if line.startswith('__label__'):
                parts = line.split(' ', 1)
                if len(parts) == 2:
                    line = parts[1].strip()
            # 处理制表符分隔
            elif '\t' in line:
                line = line.split('\t')[-1].strip()
            if self._is_natural_language(line):
                valid_lines.append(line)
        return valid_lines

    def load_data(self, data_dir, n_samples=None):
        """
        从目录加载所有文本文件，自动检测编码，过滤非自然语言行。
        优先读取自然语言相关的子目录（如 news, weibo, novel 等）。
        """
        texts = []
        priority_dirs = ['news', 'weibo', 'novel', 'story', 'baike', 'forum']
        other_files = []

        for root, dirs, files in os.walk(data_dir):
            dir_name = os.path.basename(root)
            if dir_name in priority_dirs:
                for file in files:
                    if file.endswith(('.txt', '.csv', '.json')):
                        texts.extend(self._process_file(os.path.join(root, file)))
            else:
                for file in files:
                    if file.endswith(('.txt', '.csv', '.json')):
                        other_files.append(os.path.join(root, file))

        for file_path in other_files:
            texts.extend(self._process_file(file_path))

        if self.verbose:
            print(f"原始读取行数（已过滤非自然语言）: {len(texts)}")

        if not texts:
            return []

        # 去重（加载阶段轻量去重）
        texts = list(dict.fromkeys(texts))

        if n_samples and n_samples < len(texts):
            random.shuffle(texts)
            texts = texts[:n_samples]
            if self.verbose:
                print(f"随机采样 {n_samples} 条数据")
        elif self.verbose:
            print(f"共加载 {len(texts)} 条文本")

        return texts


# ==================== 主流水线 ====================
class CorpusPipeline:
    def __init__(self, verbose=False):
        self.verbose = verbose
        self.loader = RealDataLoader(verbose=verbose)
        self.cleaner = CorpusCleaner(verbose=verbose)

    def run(self, data_dir, n_samples=None, high_threshold=2.0, output_dir='./output'):
        """
        运行清洗流水线。
        data_dir: 真实数据集目录（必须存在且包含文本文件）
        n_samples: 从真实数据集中采样数量（None表示全部使用）
        """
        if not os.path.exists(data_dir):
            raise FileNotFoundError(f"数据目录不存在: {data_dir}")

        texts = self.loader.load_data(data_dir, n_samples)
        if not texts:
            raise ValueError("未从数据目录中读取到任何有效文本，请检查目录内容或自然语言过滤条件。")

        df = pd.DataFrame({'text': texts, 'source': 'real'})
        if self.verbose:
            print(f"真实数据加载完成，总计 {len(df)} 条")

        df = self.cleaner.exact_deduplication(df)
        df = self.cleaner.desensitize_text(df)
        df = self.cleaner.quality_scoring(df)
        high, low = self.cleaner.generate_report(df, high_threshold, output_dir)
        return df, high, low


# ==================== 运行示例 ====================
if __name__ == "__main__":
    # 本地数据集路径（请根据实际路径修改）
    data_path = r"./中文清洗数据集/Small-Chinese-Corpus-master"

    pipeline = CorpusPipeline(verbose=True)
    # 使用真实数据集，采样 30000 条（若数据不足则使用全部）
    df, high, low = pipeline.run(data_dir=data_path, n_samples=30000, high_threshold=2.0)
    print(f"\n最终结果：高质量 {len(high)} 条，低质量 {len(low)} 条")